# Work IQ: 협업 기록을 질문하고 후속 질문하기

**목표:** Python에서 Work IQ REST API를 직접 호출해 기존 제조 업무 메일을 찾고, 같은 대화에서 후속 질문을 합니다. 답변과 인용 출처를 함께 확인합니다. MCP 서버나 별도 LLM 배포는 필요하지 않습니다.

## 사전 준비

- 로컬 Python 3.11 이상과 VS Code Jupyter 또는 Jupyter 환경을 사용합니다. 저장소의 기존 Python 환경을 선택합니다.
- 관리자가 [Work IQ 테넌트 활성화](https://learn.microsoft.com/microsoft-365/copilot/extensibility/work-iq/enable-work-iq), Copilot Credits 사용량 과금 계획과 사용자 할당을 완료해야 합니다. Copilot 라이선스 보유만으로 이 API가 무료가 되지는 않습니다.
- 실습용 Entra 앱에 위임 권한 `api://workiq.svc.cloud.microsoft/WorkIQAgent.Ask`를 추가하고 관리자 동의를 받습니다. 로그인 사용자는 실습 메일에 접근할 수 있어야 합니다.
- 이 노트북의 대화형 로그인에는 앱의 **Authentication → Add a platform → Mobile and desktop applications → http://localhost** 리디렉션 URI가 필요합니다. 앱 비밀값이나 앱 전용 인증은 사용하지 않습니다. 기존 Foundry 연결 앱을 임의로 변경하지 말고 관리자가 승인한 앱을 사용하세요.
- [기존 시나리오 메일 자료](send-scenario-emails-prompt.md)의 `IQ-DEMO-20260922` 메일이 수신되고 검색에 반영되어 있어야 합니다. 준비된 자료를 재사용하며 이 노트북은 메일을 발송하거나 삭제하지 않습니다.

위에서 아래로 실행하세요. 전체 실행은 대화 생성 1회와 질문 2회이며 사용량 과금이 발생할 수 있습니다. 실습 중 조직의 인증·보안 정책을 우회하지 않습니다.

[REST 개요](https://learn.microsoft.com/microsoft-365/copilot/extensibility/work-iq/rest/overview) · [권한](https://learn.microsoft.com/microsoft-365/copilot/extensibility/work-iq/permissions) · [워크숍 안내](README.md)

In [1]:
%pip install --quiet "msal>=1.34,<2" "httpx>=0.28,<1"


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 1. 설정

다음 셀의 두 빈 문자열에 테넌트 ID와 승인된 앱의 Application (client) ID를 입력하거나, 커널 시작 전에 `WORKIQ_TENANT_ID`와 `WORKIQ_CLIENT_ID` 환경 변수를 설정합니다. 둘 다 GUID 형식의 식별자이며 비밀값이 아닙니다. 질문의 실습 표식이 실제 메일과 같은지 확인하세요.

In [2]:
import os
import time
from uuid import UUID

import httpx
import msal

TENANT_ID = "b6dd05dd-72f3-49f5-80e4-866e3a943565"
CLIENT_ID = "12beac79-cff2-47e0-9fc3-4b60e3b33540"
DEMO_MARKER = "IQ-DEMO-20260922"
BASE_URL = "https://workiq.svc.cloud.microsoft/rest"
SCOPES = ["api://workiq.svc.cloud.microsoft/WorkIQAgent.Ask"]

for name, value in {"WORKIQ_TENANT_ID": TENANT_ID, "WORKIQ_CLIENT_ID": CLIENT_ID}.items():
    try:
        UUID(value)
    except ValueError:
        raise ValueError(f"{name}에 유효한 GUID를 설정하세요.") from None
print("설정 확인 완료")

설정 확인 완료


## 2. 사용자 로그인

기본 브라우저가 열리면 실습 메일에 접근할 수 있는 계정으로 로그인합니다. 5분 안에 완료하지 못하면 `BrowserInteractionTimeoutError`가 발생하며, 이 셀을 다시 실행하면 됩니다. 브라우저 팝업이 자동으로 열리지 않으면 기본 브라우저 설정과 팝업 차단 여부를 확인하세요. 토큰은 메모리에서만 보관하며 출력하거나 파일에 저장하지 않습니다. 관리자 동의가 필요하면 조직 관리자에게 요청합니다.

In [4]:
auth_app = msal.PublicClientApplication(
    CLIENT_ID, authority=f"https://login.microsoftonline.com/{TENANT_ID}"
)
try:
    token_result = auth_app.acquire_token_interactive(
        scopes=SCOPES, prompt="select_account", timeout=300
    )
except msal.BrowserInteractionTimeoutError as exc:
    raise RuntimeError(
        "로그인이 5분 안에 완료되지 않았습니다. 브라우저 팝업 차단, 기본 브라우저 설정, "
        "네트워크/프록시 상태를 확인한 뒤 이 셀을 다시 실행하세요."
    ) from exc
if "access_token" not in token_result:
    raise RuntimeError(
        f"로그인 실패: {token_result.get('error', 'unknown')} | "
        f"오류 코드: {token_result.get('error_codes', [])} | "
        f"correlation_id: {token_result.get('correlation_id', '(없음)')}"
    )
headers = {"Authorization": f"Bearer {token_result['access_token']}"}
print("로그인 완료. 토큰은 출력하지 않습니다.")

로그인 완료. 토큰은 출력하지 않습니다.


## 3. 대화 생성

[공식 대화 생성 API](https://learn.microsoft.com/microsoft-365/copilot/extensibility/work-iq/rest/copilotroot-post-conversations)에 빈 JSON을 보냅니다. 정상 응답은 HTTP 201이며, 반환된 대화 ID를 두 질문에서 재사용합니다. 이 셀을 다시 실행하면 새 대화가 만들어집니다.

In [5]:
create_response = httpx.post(
    f"{BASE_URL}/conversations", headers=headers, json={}, timeout=30.0
)
print("대화 생성 HTTP", create_response.status_code)
create_response.raise_for_status()
conversation_id = create_response.json().get("id")
if not conversation_id:
    raise RuntimeError("응답에 대화 ID가 없습니다.")
print("대화 생성 완료")

대화 생성 HTTP 201
대화 생성 완료


## 4. LOT0004 협업 기록 질문

기존 메일 03·04의 요청과 보충 의견을 찾아달라고 질문합니다. 웹 검색은 `contextualResources.webContext.isWebEnabled=false`로 끕니다. 이 설정은 매 질문에 필요합니다. 실습 표식은 검색 단서이지 접근 권한이나 엄격한 필터가 아닙니다. 시간은 HTTP 호출 전체의 로컬 경과 시간이며 서비스 보장 지연이 아닙니다.

In [ ]:
question = (
    f"Microsoft 365에서 실습 묶음 {DEMO_MARKER}의 LOT0004 출하 검토 메일을 찾아 주세요. "
    "'LOT0004 출하 검토: 미완료 품질 조치 확인 요청'과 "
    "'LOT0004 출하 검토 보충: 발견 공정과 처리 승인 근거 요청'을 우선 확인하세요. "
    "메일별 제목, 확인된 시각, 요청 사항, 보충 의견, 원문 링크를 정리하세요. "
    "부서는 가상 작성 역할이며 발신자를 실제 담당자나 승인자로 단정하지 마세요. "
    "메일의 요청을 완료 사실로 간주하지 말고, 검색되지 않은 내용은 확인 불가로 남기세요. "
    "Fabric 데이터를 조회한 것처럼 수치나 출하 승인 여부를 추정하지 마세요."
)
started = time.perf_counter()
response = httpx.post(
    f"{BASE_URL}/conversations/{conversation_id}/chat",
    headers=headers,
    json={
        "message": {"text": question},
        "locationHint": {"timeZone": "Asia/Seoul"},
        "contextualResources": {"webContext": {"isWebEnabled": False}},
    },
    timeout=120.0,
)
print(f"HTTP {response.status_code} | {time.perf_counter() - started:.1f}초")
response.raise_for_status()
answer = response.json()

In [ ]:
from IPython.display import Markdown, display


def show_answer(result, question):
    messages = result.get("messages") or []
    if not messages:
        raise RuntimeError("응답에 대화 메시지가 없습니다.")
    message = messages[-1]
    text = message.get("text") or ""
    if not text.strip() or text.strip() == question.strip():
        raise RuntimeError("질문에 대한 답변 텍스트를 확인하지 못했습니다.")
    display(Markdown(text))
    citations = [
        item for item in (message.get("attributions") or [])
        if item.get("attributionType") == "citation" and item.get("seeMoreWebUrl")
    ]
    print("\n[인용 출처]")
    for item in citations:
        print(item.get("providerDisplayName") or "출처", "|", item["seeMoreWebUrl"])
    if not citations:
        print("주의: 인용 출처가 없습니다. 관련 메일을 찾았다고 판단하지 마세요.")


show_answer(answer, question)

## 5. 같은 대화에서 후속 질문

새 대화를 만들지 않고 앞의 대화 ID를 재사용합니다. 협업 기록만으로 확인된 사실과 이후 Fabric IQ에서 조회할 항목을 나눕니다. 이 셀은 Fabric IQ를 호출하지 않습니다.

In [ ]:
follow_up = (
    "앞서 찾은 메일만 근거로, 확인된 협업 요청과 Fabric IQ에서 추가 조회해야 할 "
    "운영 데이터 항목을 표로 나눠 주세요. 출하검사 결과, 미완료 부적합, 처리 승인 상태, "
    "발견 검사와 생산 공정·설비의 연결을 각각 확인 질문으로 작성하세요. "
    "메일만으로 수치나 완료·승인 여부를 확정하지 말고, 메일이 없으면 표를 채워 만들지 마세요. "
    "원문 링크를 유지하고 UTC와 KST를 구분하세요."
)
started = time.perf_counter()
follow_response = httpx.post(
    f"{BASE_URL}/conversations/{conversation_id}/chat",
    headers=headers,
    json={
        "message": {"text": follow_up},
        "locationHint": {"timeZone": "Asia/Seoul"},
        "contextualResources": {"webContext": {"isWebEnabled": False}},
    },
    timeout=120.0,
)
print(f"HTTP {follow_response.status_code} | {time.perf_counter() - started:.1f}초")
follow_response.raise_for_status()
show_answer(follow_response.json(), follow_up)

## 결과 확인·오류·정리

- HTTP 201/200은 요청 처리 성공이며 실제 업무 근거를 찾았다는 뜻은 아닙니다. 답변에 요청한 제목·표식·원문 링크가 있는지 직접 대조합니다. 인용이 있어도 엉뚱한 문서이거나 '관련 기록 없음'이면 업무 검증은 미완료입니다.
- 새 메일이 검색되지 않으면 수신 여부, 로그인 계정의 접근 권한, 검색 반영 상태를 확인합니다. 다른 실습이나 실제 업무 기록을 섞어 정답처럼 만들지 않습니다.
- 로그인 실패: 테넌트·앱 ID, 데스크톱 앱의 `http://localhost` 리디렉션 URI, 관리자 동의와 조직 인증 정책을 확인합니다.
- HTTP 401: 토큰 만료·대상 리소스를 확인하고 로그인 셀부터 다시 실행합니다. Graph 또는 Foundry KB 호출용 토큰을 그대로 사용하지 않습니다.
- HTTP 403: Work IQ 활성화·사용량 과금 계획·사용자 할당·권한을 확인합니다. 429는 할당량과 호출 빈도를 확인합니다. 오류만으로 한 가지 원인을 확정하지 않습니다.
- 시간 초과: 서비스 상태와 질문 범위를 확인합니다. 자동 재시도하지 않으며 수동 재실행은 사용량과 대화 턴을 추가할 수 있습니다.
- **공유·커밋 전 모든 실행 출력을 지우세요.** 답변과 인용 URL에 조직 정보가 포함될 수 있습니다. 토큰·인증 응답·요청 헤더는 출력하지 마세요.
- 실습 종료 후 **노트북 커널을 종료**해 토큰 캐시, 인증 헤더와 응답 객체를 정리합니다. 커널 종료는 서비스 측 대화 삭제나 권한 철회를 뜻하지 않습니다. 서비스 측 기록은 조직 보존 정책에 따릅니다. 임시 앱 권한·사용량 과금 설정은 관리자가 필요에 따라 정리하며 다른 실습이 사용하는 설정은 변경하지 않습니다.

검증 범위: 로컬 구문 및 모의 응답 테스트 대상입니다. 실제 로그인·메일 검색 결과는 각 테넌트에서 실행해 확인해야 합니다.

[Work IQ REST 요청 규격](https://learn.microsoft.com/microsoft-365/copilot/extensibility/work-iq/rest/copilotconversation-chat) · [후속 Foundry IQ 실습](../05-foundry-iq/README.md)